#Proposta

Criar modelos de classificação em cima de dados de marketing, para classificar o risco de churn

##Instalando e iniciando uma sessão Spark

In [1]:
# Install Java
!apt-get update -qq > /dev/null
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Download and extract Spark
!wget -q http://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz

# Install PySpark
!pip install pyspark==3.5.0

# Set environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Initialize SparkSession
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425346 sha256=78f480bff3ab2e06956fb5f2517364f14a4c5cfabb55f828e8c78eac0f463101
  Stored in directory: /root/.cache/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.1
    Uninstalling pyspark-3.5.1:
      Successfully uninstalled pyspark-3.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, b

In [2]:
spark

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dados = spark.read.csv("/content/drive/MyDrive/Curso_Spark/base de dados/base de dados/dados_clientes.csv", sep=",", header= True, inferSchema=True)

In [5]:
dados

DataFrame[id: int, Churn: string, Mais65anos: int, Conjuge: string, Dependentes: string, MesesDeContrato: int, TelefoneFixo: string, MaisDeUmaLinhaTelefonica: string, Internet: string, SegurancaOnline: string, BackupOnline: string, SeguroDispositivo: string, SuporteTecnico: string, TVaCabo: string, StreamingFilmes: string, TipoContrato: string, ContaCorreio: string, MetodoPagamento: string, MesesCobrados: double]

In [6]:
import pandas as pd

In [7]:
dados.show()

+---+-----+----------+-------+-----------+---------------+------------+------------------------+-----------+------------------+------------------+------------------+------------------+------------------+------------------+------------+------------+----------------+-------------+
| id|Churn|Mais65anos|Conjuge|Dependentes|MesesDeContrato|TelefoneFixo|MaisDeUmaLinhaTelefonica|   Internet|   SegurancaOnline|      BackupOnline| SeguroDispositivo|    SuporteTecnico|           TVaCabo|   StreamingFilmes|TipoContrato|ContaCorreio| MetodoPagamento|MesesCobrados|
+---+-----+----------+-------+-----------+---------------+------------+------------------------+-----------+------------------+------------------+------------------+------------------+------------------+------------------+------------+------------+----------------+-------------+
|  0|  Nao|         0|    Sim|        Nao|              1|         Nao|    SemServicoTelefonico|        DSL|               Nao|               Sim|              

In [8]:
dados.limit(5).toPandas()

,id,Churn,Mais65anos,Conjuge,Dependentes,MesesDeContrato,TelefoneFixo,MaisDeUmaLinhaTelefonica,Internet,SegurancaOnline,BackupOnline,SeguroDispositivo,SuporteTecnico,TVaCabo,StreamingFilmes,TipoContrato,ContaCorreio,MetodoPagamento,MesesCobrados
0,0,Nao,0,Sim,Nao,1,Nao,SemServicoTelefonico,DSL,Nao,Sim,Nao,Nao,Nao,Nao,Mensalmente,Sim,BoletoEletronico,29.85
1,1,Nao,0,Nao,Nao,34,Sim,Nao,DSL,Sim,Nao,Sim,Nao,Nao,Nao,UmAno,Nao,Boleto,56.95
2,2,Sim,0,Nao,Nao,2,Sim,Nao,DSL,Sim,Sim,Nao,Nao,Nao,Nao,Mensalmente,Sim,Boleto,53.85
3,3,Nao,0,Nao,Nao,45,Nao,SemServicoTelefonico,DSL,Sim,Nao,Sim,Sim,Nao,Nao,UmAno,Nao,DebitoEmConta,42.30
4,4,Sim,0,Nao,Nao,2,Sim,Nao,FibraOptica,Nao,Nao,Nao,Nao,Nao,Nao,Mensalmente,Sim,BoletoEletronico,70.70


Contando a quantidade de registros

In [9]:
dados.count()

10348

verificando o balanceamento dos dados

In [10]:
dados.groupby('Churn').count().show()

+-----+-----+
|Churn|count|
+-----+-----+
|  Sim| 5174|
|  Nao| 5174|
+-----+-----+



In [11]:
dados.printSchema()

root
 |-- id: integer (nullable = true)
 |-- Churn: string (nullable = true)
 |-- Mais65anos: integer (nullable = true)
 |-- Conjuge: string (nullable = true)
 |-- Dependentes: string (nullable = true)
 |-- MesesDeContrato: integer (nullable = true)
 |-- TelefoneFixo: string (nullable = true)
 |-- MaisDeUmaLinhaTelefonica: string (nullable = true)
 |-- Internet: string (nullable = true)
 |-- SegurancaOnline: string (nullable = true)
 |-- BackupOnline: string (nullable = true)
 |-- SeguroDispositivo: string (nullable = true)
 |-- SuporteTecnico: string (nullable = true)
 |-- TVaCabo: string (nullable = true)
 |-- StreamingFilmes: string (nullable = true)
 |-- TipoContrato: string (nullable = true)
 |-- ContaCorreio: string (nullable = true)
 |-- MetodoPagamento: string (nullable = true)
 |-- MesesCobrados: double (nullable = true)



##Tratando os dados

In [12]:
colunasBinarias =  [
    'Churn',
    'Conjuge',
    'Dependentes',
    'TelefoneFixo',
    'MaisDeUmaLinhaTelefonica',
    'SegurancaOnline',
    'BackupOnline',
    'SeguroDispositivo',
    'SuporteTecnico',
    'TVaCabo',
    'StreamingFilmes',
    'ContaCorreio'
]

In [13]:
from pyspark.sql import functions as f

Criando uma lista de expressões para processar as colunas binárias do seu DataFrame PySpark. Para cada coluna listada em colunasBinarias, ele verifica se o valor da coluna é 'Sim'. Se for, ele atribui o valor 1; caso contrário, atribui 0. Isso é feito para converter as respostas textuais ('Sim'/'Nao') em um formato numérico (1/0), que é mais adequado para a construção de modelos de classificação.



In [14]:
todasColunas = [
    f.when(f.col(c) =='Sim', 1).otherwise(0).alias(c) for c in colunasBinarias
]

vendo a Syntax/regra criada para cada coluna binária

In [15]:
todasColunas

[Column<'CASE WHEN (Churn = Sim) THEN 1 ELSE 0 END AS Churn'>,
 Column<'CASE WHEN (Conjuge = Sim) THEN 1 ELSE 0 END AS Conjuge'>,
 Column<'CASE WHEN (Dependentes = Sim) THEN 1 ELSE 0 END AS Dependentes'>,
 Column<'CASE WHEN (TelefoneFixo = Sim) THEN 1 ELSE 0 END AS TelefoneFixo'>,
 Column<'CASE WHEN (MaisDeUmaLinhaTelefonica = Sim) THEN 1 ELSE 0 END AS MaisDeUmaLinhaTelefonica'>,
 Column<'CASE WHEN (SegurancaOnline = Sim) THEN 1 ELSE 0 END AS SegurancaOnline'>,
 Column<'CASE WHEN (BackupOnline = Sim) THEN 1 ELSE 0 END AS BackupOnline'>,
 Column<'CASE WHEN (SeguroDispositivo = Sim) THEN 1 ELSE 0 END AS SeguroDispositivo'>,
 Column<'CASE WHEN (SuporteTecnico = Sim) THEN 1 ELSE 0 END AS SuporteTecnico'>,
 Column<'CASE WHEN (TVaCabo = Sim) THEN 1 ELSE 0 END AS TVaCabo'>,
 Column<'CASE WHEN (StreamingFilmes = Sim) THEN 1 ELSE 0 END AS StreamingFilmes'>,
 Column<'CASE WHEN (ContaCorreio = Sim) THEN 1 ELSE 0 END AS ContaCorreio'>]

Agora vamos inserir as colunas não binárias

 percorrendo as colunas do seu DataFrame dados de trás para frente. Para cada coluna, ele verifica se o nome da coluna NÃO está presente na lista colunasBinarias. Se a coluna não for binária, ela é inserida no início (índice 0) da lista todasColunas.

In [16]:
for coluna in reversed(dados.columns):
  if coluna not in colunasBinarias:
    todasColunas.insert(0, coluna)

In [17]:
todasColunas

['id',
 'Mais65anos',
 'MesesDeContrato',
 'Internet',
 'TipoContrato',
 'MetodoPagamento',
 'MesesCobrados',
 Column<'CASE WHEN (Churn = Sim) THEN 1 ELSE 0 END AS Churn'>,
 Column<'CASE WHEN (Conjuge = Sim) THEN 1 ELSE 0 END AS Conjuge'>,
 Column<'CASE WHEN (Dependentes = Sim) THEN 1 ELSE 0 END AS Dependentes'>,
 Column<'CASE WHEN (TelefoneFixo = Sim) THEN 1 ELSE 0 END AS TelefoneFixo'>,
 Column<'CASE WHEN (MaisDeUmaLinhaTelefonica = Sim) THEN 1 ELSE 0 END AS MaisDeUmaLinhaTelefonica'>,
 Column<'CASE WHEN (SegurancaOnline = Sim) THEN 1 ELSE 0 END AS SegurancaOnline'>,
 Column<'CASE WHEN (BackupOnline = Sim) THEN 1 ELSE 0 END AS BackupOnline'>,
 Column<'CASE WHEN (SeguroDispositivo = Sim) THEN 1 ELSE 0 END AS SeguroDispositivo'>,
 Column<'CASE WHEN (SuporteTecnico = Sim) THEN 1 ELSE 0 END AS SuporteTecnico'>,
 Column<'CASE WHEN (TVaCabo = Sim) THEN 1 ELSE 0 END AS TVaCabo'>,
 Column<'CASE WHEN (StreamingFilmes = Sim) THEN 1 ELSE 0 END AS StreamingFilmes'>,
 Column<'CASE WHEN (ContaCorr

visualizando as colunas/dados binários transformados

In [18]:
dados.select(todasColunas).show()

+---+----------+---------------+-----------+------------+----------------+-------------+-----+-------+-----------+------------+------------------------+---------------+------------+-----------------+--------------+-------+---------------+------------+
| id|Mais65anos|MesesDeContrato|   Internet|TipoContrato| MetodoPagamento|MesesCobrados|Churn|Conjuge|Dependentes|TelefoneFixo|MaisDeUmaLinhaTelefonica|SegurancaOnline|BackupOnline|SeguroDispositivo|SuporteTecnico|TVaCabo|StreamingFilmes|ContaCorreio|
+---+----------+---------------+-----------+------------+----------------+-------------+-----+-------+-----------+------------+------------------------+---------------+------------+-----------------+--------------+-------+---------------+------------+
|  0|         0|              1|        DSL| Mensalmente|BoletoEletronico|        29.85|    0|      1|          0|           0|                       0|              0|           1|                0|             0|      0|              0|      

In [19]:
dataset = dados.select(todasColunas)

#Criando Dummies

In [20]:
dataset.printSchema()

root
 |-- id: integer (nullable = true)
 |-- Mais65anos: integer (nullable = true)
 |-- MesesDeContrato: integer (nullable = true)
 |-- Internet: string (nullable = true)
 |-- TipoContrato: string (nullable = true)
 |-- MetodoPagamento: string (nullable = true)
 |-- MesesCobrados: double (nullable = true)
 |-- Churn: integer (nullable = false)
 |-- Conjuge: integer (nullable = false)
 |-- Dependentes: integer (nullable = false)
 |-- TelefoneFixo: integer (nullable = false)
 |-- MaisDeUmaLinhaTelefonica: integer (nullable = false)
 |-- SegurancaOnline: integer (nullable = false)
 |-- BackupOnline: integer (nullable = false)
 |-- SeguroDispositivo: integer (nullable = false)
 |-- SuporteTecnico: integer (nullable = false)
 |-- TVaCabo: integer (nullable = false)
 |-- StreamingFilmes: integer (nullable = false)
 |-- ContaCorreio: integer (nullable = false)



In [21]:
dados.select(['Internet', 'TipoContrato', 'MetodoPagamento']).show()

+-----------+------------+----------------+
|   Internet|TipoContrato| MetodoPagamento|
+-----------+------------+----------------+
|        DSL| Mensalmente|BoletoEletronico|
|        DSL|       UmAno|          Boleto|
|        DSL| Mensalmente|          Boleto|
|        DSL|       UmAno|   DebitoEmConta|
|FibraOptica| Mensalmente|BoletoEletronico|
|FibraOptica| Mensalmente|BoletoEletronico|
|FibraOptica| Mensalmente|   CartaoCredito|
|        DSL| Mensalmente|          Boleto|
|FibraOptica| Mensalmente|BoletoEletronico|
|        DSL|       UmAno|   DebitoEmConta|
|        DSL| Mensalmente|          Boleto|
|        Nao|    DoisAnos|   CartaoCredito|
|FibraOptica|       UmAno|   CartaoCredito|
|FibraOptica| Mensalmente|   DebitoEmConta|
|FibraOptica| Mensalmente|BoletoEletronico|
|FibraOptica|    DoisAnos|   CartaoCredito|
|        Nao|       UmAno|          Boleto|
|FibraOptica|    DoisAnos|   DebitoEmConta|
|        DSL| Mensalmente|   CartaoCredito|
|FibraOptica| Mensalmente|Boleto

In [22]:
dados.groupBy('Internet').count().show()

+-----------+-----+
|   Internet|count|
+-----------+-----+
|FibraOptica| 5401|
|        Nao| 1741|
|        DSL| 3206|
+-----------+-----+



In [23]:
dados.groupBy('TipoContrato').count().show()

+------------+-----+
|TipoContrato|count|
+------------+-----+
|       UmAno| 1672|
| Mensalmente| 6926|
|    DoisAnos| 1750|
+------------+-----+



In [24]:
dados.groupBy('MetodoPagamento').count().show()

+----------------+-----+
| MetodoPagamento|count|
+----------------+-----+
|BoletoEletronico| 4714|
|   CartaoCredito| 1761|
|   DebitoEmConta| 1822|
|          Boleto| 2051|
+----------------+-----+



 realizando uma operação de 'pivot' na sua coluna 'Internet' dentro do DataFrame dataset. Vamos por partes:

dataset.groupby('id'): Primeiro, ele agrupa os dados pelo 'id' de cada cliente.

.pivot('Internet'): Em seguida, ele 'pivota' o DataFrame usando a coluna 'Internet'. Isso significa que cada valor único na coluna 'Internet' (como 'DSL', 'FibraOptica', 'Nao') se tornará uma nova coluna no DataFrame resultante.

.agg(f.lit(1)): Para cada cliente ('id') e tipo de internet, ele atribui o valor 1. Ou seja, se um cliente tem 'DSL', na nova coluna 'DSL' para aquele cliente, o valor será 1.


.na.fill(0): Como resultado do pivot, os clientes que não possuem um determinado tipo de internet terão valores nulos nas colunas correspondentes. Este comando preenche esses valores nulos com 0.


.show(): Finalmente, ele exibe as primeiras 20 linhas do DataFrame resultante.


Em essência, este código está criando colunas 'dummy' ou fazendo um 'one-hot encoding' para a coluna 'Internet',

In [25]:
dataset.groupby('id').pivot('Internet').agg(f.lit(1)).na.fill(0).show()

+----+---+-----------+---+
|  id|DSL|FibraOptica|Nao|
+----+---+-----------+---+
|7982|  1|          0|  0|
|9465|  0|          1|  0|
|2122|  1|          0|  0|
|3997|  1|          0|  0|
|6654|  0|          1|  0|
|7880|  0|          1|  0|
|4519|  0|          1|  0|
|6466|  0|          1|  0|
| 496|  1|          0|  0|
|7833|  0|          1|  0|
|1591|  0|          0|  1|
|2866|  0|          1|  0|
|8592|  0|          1|  0|
|1829|  0|          1|  0|
| 463|  0|          1|  0|
|4900|  0|          1|  0|
|4818|  0|          1|  0|
|7554|  1|          0|  0|
|1342|  0|          0|  1|
|5300|  0|          1|  0|
+----+---+-----------+---+
only showing top 20 rows



Criando os dummies para cada coluna

In [26]:
Internet = dataset.groupby('id').pivot('Internet').agg(f.lit(1)).na.fill(0)
TipoContrato = dataset.groupby('id').pivot('TipoContrato').agg(f.lit(1)).na.fill(0)
MetodoPagamento = dataset.groupby('id').pivot('MetodoPagamento').agg(f.lit(1)).na.fill(0)

.join(Internet, 'id', how='inner'): Primeiro, ele une o dataset com o DataFrame Internet (que contém as colunas DSL, FibraOptica, Nao) usando a coluna id como chave. Um inner join significa que apenas as linhas que têm id correspondente em ambos os DataFrames serão mantidas.


.join(TipoContrato, 'id', how='inner'): Em seguida, o resultado da junção anterior é unido com o DataFrame TipoContrato (com as colunas DoisAnos, Mensalmente, UmAno), novamente usando id.


.join(MetodoPagamento, 'id', how='inner'): O mesmo processo é feito para o DataFrame MetodoPagamento (com as colunas dos métodos de pagamento).


.select(...): Após todas as junções, esta parte do código seleciona as colunas para o novo DataFrame resultante:

'*': Inclui todas as colunas que já existiam no dataset original e nas colunas id dos DataFrames de dummy.


f.col('DSL').alias('Internet_DSL'), f.col('FibraOptica').alias('Internet_FibraOptica'), etc.: Estas linhas selecionam as novas colunas 'dummy' que foram adicionadas pelas junções e as renomeiam para torná-las mais claras e específicas (por exemplo, DSL se torna Internet_DSL), evitando ambiguidades e facilitando a compreensão do que cada coluna representa.

(.drop): Por fim, ele remove as colunas originais que foram transformadas (Internet, TipoContrato, MetodoPagamento) e as colunas intermediárias (como DSL, FibraOptica, etc., que foram as colunas temporárias geradas pelo pivot antes da renomeação). Isso limpa o DataFrame, deixando apenas as novas variáveis one-hot encoded e as colunas originais que não foram transformadas.

In [27]:
dataset\
.join(Internet, 'id', how='inner')\
.join(TipoContrato, 'id', how='inner')\
.join(MetodoPagamento, 'id', how='inner')\
.select(
    '*',
    f.col('DSL').alias('Internet_DSL)'),
    f.col('FibraOptica').alias('Internet_FibraOptica'),
    f.col('Nao').alias('Internet_Nao'),
    f.col('Mensalmente').alias('TipoContrato_Mensalmente'),
    f.col('UmAno').alias('TipoContrato_UmAno'),
    f.col('DoisAnos').alias('TipoContrato_DoisAnos'),
    f.col('DebitoEmConta').alias('MetodoPagamento_DebitoEmConta'),
    f.col('CartaoCredito').alias('MetodoPagamento_CartaoCredito'),
    f.col('BoletoEletronico').alias('MetodoPagamento_BoletoEletronico'),
    f.col('Boleto').alias('MetodoPagamento_Boleto')
)\
.drop('Internet', 'TipoContrato', 'MetodoPagamento', 'DSL', 'FibraOptica', 'Nao', 'Mensalmente', 'UmAno', 'DoisAnos', 'DebitoEmConta', 'CartaoCredito', 'BoletoEletronico', 'Boleto')\
.show()

+----+----------+---------------+-----------------+-----+-------+-----------+------------+------------------------+---------------+------------+-----------------+--------------+-------+---------------+------------+-------------+--------------------+------------+------------------------+------------------+---------------------+-----------------------------+-----------------------------+--------------------------------+----------------------+
|  id|Mais65anos|MesesDeContrato|    MesesCobrados|Churn|Conjuge|Dependentes|TelefoneFixo|MaisDeUmaLinhaTelefonica|SegurancaOnline|BackupOnline|SeguroDispositivo|SuporteTecnico|TVaCabo|StreamingFilmes|ContaCorreio|Internet_DSL)|Internet_FibraOptica|Internet_Nao|TipoContrato_Mensalmente|TipoContrato_UmAno|TipoContrato_DoisAnos|MetodoPagamento_DebitoEmConta|MetodoPagamento_CartaoCredito|MetodoPagamento_BoletoEletronico|MetodoPagamento_Boleto|
+----+----------+---------------+-----------------+-----+-------+-----------+------------+--------------------

In [28]:
dataset = dataset\
.join(Internet, 'id', how='inner')\
.join(TipoContrato, 'id', how='inner')\
.join(MetodoPagamento, 'id', how='inner')\
.select(
    '*',
    f.col('DSL').alias('Internet_DSL)'),
    f.col('FibraOptica').alias('Internet_FibraOptica'),
    f.col('Nao').alias('Internet_Nao'),
    f.col('Mensalmente').alias('TipoContrato_Mensalmente'),
    f.col('UmAno').alias('TipoContrato_UmAno'),
    f.col('DoisAnos').alias('TipoContrato_DoisAnos'),
    f.col('DebitoEmConta').alias('MetodoPagamento_DebitoEmConta'),
    f.col('CartaoCredito').alias('MetodoPagamento_CartaoCredito'),
    f.col('BoletoEletronico').alias('MetodoPagamento_BoletoEletronico'),
    f.col('Boleto').alias('MetodoPagamento_Boleto')
)\
.drop('Internet', 'TipoContrato', 'MetodoPagamento', 'DSL', 'FibraOptica', 'Nao', 'Mensalmente', 'UmAno', 'DoisAnos', 'DebitoEmConta', 'CartaoCredito', 'BoletoEletronico', 'Boleto')

In [29]:
dataset.show()

+----+----------+---------------+-----------------+-----+-------+-----------+------------+------------------------+---------------+------------+-----------------+--------------+-------+---------------+------------+-------------+--------------------+------------+------------------------+------------------+---------------------+-----------------------------+-----------------------------+--------------------------------+----------------------+
|  id|Mais65anos|MesesDeContrato|    MesesCobrados|Churn|Conjuge|Dependentes|TelefoneFixo|MaisDeUmaLinhaTelefonica|SegurancaOnline|BackupOnline|SeguroDispositivo|SuporteTecnico|TVaCabo|StreamingFilmes|ContaCorreio|Internet_DSL)|Internet_FibraOptica|Internet_Nao|TipoContrato_Mensalmente|TipoContrato_UmAno|TipoContrato_DoisAnos|MetodoPagamento_DebitoEmConta|MetodoPagamento_CartaoCredito|MetodoPagamento_BoletoEletronico|MetodoPagamento_Boleto|
+----+----------+---------------+-----------------+-----+-------+-----------+------------+--------------------

#Preparação dos dados para treinamento dos modelos

In [30]:
from pyspark.ml.feature import VectorAssembler

Renomeando as colunas para label e features, para usar a biblioteca pyspark.ml

In [31]:
dataset = dataset.withColumnRenamed('Churn', 'label')

In [32]:
X = dataset.columns
X.remove('label')
X.remove('id')
X

['Mais65anos',
 'MesesDeContrato',
 'MesesCobrados',
 'Conjuge',
 'Dependentes',
 'TelefoneFixo',
 'MaisDeUmaLinhaTelefonica',
 'SegurancaOnline',
 'BackupOnline',
 'SeguroDispositivo',
 'SuporteTecnico',
 'TVaCabo',
 'StreamingFilmes',
 'ContaCorreio',
 'Internet_DSL)',
 'Internet_FibraOptica',
 'Internet_Nao',
 'TipoContrato_Mensalmente',
 'TipoContrato_UmAno',
 'TipoContrato_DoisAnos',
 'MetodoPagamento_DebitoEmConta',
 'MetodoPagamento_CartaoCredito',
 'MetodoPagamento_BoletoEletronico',
 'MetodoPagamento_Boleto']

In [33]:
assembler = VectorAssembler(inputCols=X, outputCol='features')

Entendendo o assembler

24 é o número de features

a primeira lista são as colunas que com valores

a segunda lista são os valores das colunas identificada na primeira lista

In [34]:
assembler.transform(dataset).select('features', 'label').show(10, truncate = False)

+-----------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                   |label|
+-----------------------------------------------------------------------------------------------------------+-----+
|(24,[1,2,11,12,13,14,17,22],[1.0,45.30540797610398,1.0,1.0,1.0,1.0,1.0,1.0])                               |1    |
|(24,[1,2,3,5,6,8,9,11,12,13,15,17,22],[60.0,103.6142230120257,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|1    |
|(24,[1,2,5,6,10,11,12,13,14,18,23],[12.0,75.85,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])                       |0    |
|(24,[1,2,3,5,8,12,13,14,19,21],[69.0,61.45,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])                               |0    |
|(24,[1,2,3,5,6,11,13,15,17,22],[7.0,86.5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])                                 |1    |
|(24,[1,2,5,6,12,13,15,17,22],[14.0,85.03742670311915,1.0,1.0,1.0,1.0,1.

In [35]:
dataset_prep = assembler.transform(dataset).select('features', 'label')

#Ajuste e previsão

usando randomSplit para separar os dados em treino e teste

In [36]:
SEED = 101

In [37]:
treino, teste = dataset_prep.randomSplit([0.7, 0.3], seed=SEED)

In [38]:
print("treino:", treino.count())
print("teste:", teste.count())

treino: 7206
teste: 3142


In [39]:
from pyspark.ml.classification import LogisticRegression

In [40]:
lr = LogisticRegression()

treinando o modelo

In [41]:
modelo_lr = lr.fit(treino)

In [42]:
previsoes_lr_teste = modelo_lr.transform(teste)

In [43]:
previsoes_lr_teste.show()

+--------------------+-----+--------------------+--------------------+----------+
|            features|label|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|(24,[0,1,2,3,4,5,...|    0|[3.02174179751551...|[0.95354674000282...|       0.0|
|(24,[0,1,2,3,4,5,...|    0|[-0.0922192966076...|[0.47696150091605...|       1.0|
|(24,[0,1,2,3,4,5,...|    1|[0.18744121711361...|[0.54672358463156...|       0.0|
|(24,[0,1,2,3,4,5,...|    1|[0.91716501260103...|[0.71446410549163...|       0.0|
|(24,[0,1,2,3,4,5,...|    0|[-0.1495904711610...|[0.46267196467801...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[-0.1680594619286...|[0.45808374494006...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[-1.4170949608173...|[0.19511740608882...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[0.14194260698794...|[0.53542619200881...|       0.0|
|(24,[0,1,2,3,4,5,...|    0|[0.67046644011599...|[0.66160759507905...|       0.0|
|(24,[0,1,2,3,4,

#Métricas

In [44]:
resumo_lr_treino = modelo_lr.summary

In [45]:
print("Acurácia: %f" % resumo_lr_treino.accuracy)
print("Precisão: %f" % resumo_lr_treino.precisionByLabel[1])
print("Recall: %f" % resumo_lr_treino.recallByLabel[1])
print("F1: %f" % resumo_lr_treino.fMeasureByLabel()[1])

Acurácia: 0.784901
Precisão: 0.770686
Recall: 0.812517
F1: 0.791049


### Criando uma matriz de confusão

In [46]:
tp = previsoes_lr_teste.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 1)).count()
tn = previsoes_lr_teste.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 0)).count()
fp = previsoes_lr_teste.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 1)).count()
fn = previsoes_lr_teste.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 0)).count()

In [47]:
print(tp, tn, fp, fn)

1256 1179 400 307


In [48]:
print('Matriz de Confusão:')
print('             Previsão Negativa | Previsão Positiva')
print(f'Real Negativo | {tn:17d} | {fp:17d}')
print(f'Real Positivo | {fn:17d} | {tp:17d}')

Matriz de Confusão:
             Previsão Negativa | Previsão Positiva
Real Negativo |              1179 |               400
Real Positivo |               307 |              1256


Função de matriz de confusão

In [49]:
from pyspark.sql import functions as f # importo a biblioteca functions

# crio a função que vai receber os dados para serem avaliados
def calcula_mostra_metricas(modelo_lr, df_transform_modelo, normalize=False, percentage=True):
# os passos para montagem da matriz de confusão são os mesmos da aula
  tp = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 1)).count()
  tn = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 0)).count()
  fp = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 1)).count()
  fn = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 0)).count()

  valorP = 1
  valorN = 1

  if normalize:
    valorP = tp + fn
    valorN = fp + tn

  if percentage and normalize:
    valorP = valorP / 100
    valorN = valorN / 100

  # ‘s’ será minha string de retorno
  # ela vai coletar e montar minha matriz de confusão
  # e também os valores de acurácia, precisão, recall e F1-score
  s = ''

  # construção da minha string da matriz de confusão
  s += ' '*20 + 'Previsto\n'
  s += ' '*15 +  'Churn' + ' '*5 + 'Não-Churn\n'
  s += ' '*4 + 'Churn' + ' '*6 +  str(int(tp/valorP)) + ' '*7 + str(int(fn/valorP)) + '\n'
  s += 'Real\n'
  s += ' '*4 + 'Não-Churn' + ' '*2 + str(int(fp/valorN)) +  ' '*7 + str(int(tn/valorN))  + '\n'
  s += '\n'

  # coleto o resumo das métricas com summary
  resumo_lr_treino = modelo_lr.summary

  # adiciono os valores de cada métrica a minha string de retorno
  s += f'Acurácia: {resumo_lr_treino.accuracy}\n'
  s += f'Precisão: {resumo_lr_treino.precisionByLabel[1]}\n'
  s += f'Recall: {resumo_lr_treino.recallByLabel[1]}\n'
  s += f'F1: {resumo_lr_treino.fMeasureByLabel()[1]}\n'

  return s

In [50]:
print(calcula_mostra_metricas(modelo_lr, previsoes_lr_teste, normalize=True))

                    Previsto
               Churn     Não-Churn
    Churn      80       19
Real
    Não-Churn  25       74

Acurácia: 0.7849014709963918
Precisão: 0.7706855791962175
Recall: 0.8125173082248685
F1: 0.7910488002156916



#Árvore de decisão

##Ajuste e previsão

In [53]:
from pyspark.ml.classification import DecisionTreeClassifier

In [54]:
dtc = DecisionTreeClassifier(seed=SEED)

In [55]:
modelo_dtc = dtc.fit(treino)

transform é o predict

In [58]:
previsoes_dtc_treino = modelo_dtc.transform(treino)

In [56]:
previsoes_dtc_teste = modelo_dtc.transform(teste)

In [57]:
previsoes_dtc_teste.show()

+--------------------+-----+--------------+--------------------+----------+
|            features|label| rawPrediction|         probability|prediction|
+--------------------+-----+--------------+--------------------+----------+
|(24,[0,1,2,3,4,5,...|    0|[2056.0,334.0]|[0.86025104602510...|       0.0|
|(24,[0,1,2,3,4,5,...|    0|  [62.0,128.0]|[0.32631578947368...|       1.0|
|(24,[0,1,2,3,4,5,...|    1| [239.0,205.0]|[0.53828828828828...|       0.0|
|(24,[0,1,2,3,4,5,...|    1| [239.0,205.0]|[0.53828828828828...|       0.0|
|(24,[0,1,2,3,4,5,...|    0| [239.0,205.0]|[0.53828828828828...|       0.0|
|(24,[0,1,2,3,4,5,...|    0|  [51.0,141.0]| [0.265625,0.734375]|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[331.0,1951.0]|[0.14504820333041...|       1.0|
|(24,[0,1,2,3,4,5,...|    0| [239.0,205.0]|[0.53828828828828...|       0.0|
|(24,[0,1,2,3,4,5,...|    0|  [63.0,118.0]|[0.34806629834254...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[2056.0,334.0]|[0.86025104602510...|       0.0|
|(24,[0,1,2,

##Métricas

In [59]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [60]:
evaluator = MulticlassClassificationEvaluator()

In [62]:
evaluator.evaluate(previsoes_dtc_teste, {evaluator.metricName: 'accuracy'})

0.7714831317632082

Métricas Treino

In [72]:
print("Acurácia: %f" % evaluator.evaluate(previsoes_dtc_treino, {evaluator.metricName: 'accuracy'}))
print("Precisão: %f" % evaluator.evaluate(previsoes_dtc_treino, {evaluator.metricName: 'precisionByLabel'}))
print("Recall: %f" % evaluator.evaluate(previsoes_dtc_treino, {evaluator.metricName: 'recallByLabel'}))
print("F1: %f" % evaluator.evaluate(previsoes_dtc_treino, {evaluator.metricName: 'fMeasureByLabel'}))

Acurácia: 0.791701
Precisão: 0.779349
Recall: 0.812517
F1: 0.795588


Métricas Teste

In [73]:
print("Acurácia: %f" % evaluator.evaluate(previsoes_dtc_teste, {evaluator.metricName: 'accuracy'}))
print("Precisão: %f" % evaluator.evaluate(previsoes_dtc_teste, {evaluator.metricName: 'precisionByLabel'}))
print("Recall: %f" % evaluator.evaluate(previsoes_dtc_teste, {evaluator.metricName: 'recallByLabel'}))
print("F1: %f" % evaluator.evaluate(previsoes_dtc_teste, {evaluator.metricName: 'fMeasureByLabel'}))

Acurácia: 0.771483
Precisão: 0.764923
Recall: 0.787207
F1: 0.775905


Função para calcular métricas

In [74]:
from pyspark.sql import functions as f # importo a biblioteca functions
from pyspark.ml.evaluation import MulticlassClassificationEvaluator # importo a classe MulticlassClassificationEvaluator

# crio a função que vai receber os dados para serem avaliados

def calcula_mostra_metricas_evaluate(df_transform_modelo, normalize=False, percentage=True):
# os passos para montagem da matriz de confusão são os mesmos da aula
  tp = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 1)).count()
  tn = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 0)).count()
  fp = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 1)).count()
  fn = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 0)).count()

  valorP = 1
  valorN = 1

  if normalize:
    valorP = tp + fn
    valorN = fp + tn

  if percentage and normalize:
    valorP = valorP / 100
    valorN = valorN / 100

  # ‘s’ será minha string de retorno
  # ela vai coletar e montar minha matriz de confusão
  # e também os valores de acurácia, precisão, recall e F1-score
  s = ''

  # construção da minha string da matriz de confusão
  s += ' '*20 + 'Previsto\n'
  s += ' '*15 +  'Churn' + ' '*5 + 'Não-Churn\n'
  s += ' '*4 + 'Churn' + ' '*6 +  str(int(tp/valorP)) + ' '*7 + str(int(fn/valorP)) + '\n'
  s += 'Real\n'
  s += ' '*4 + 'Não-Churn' + ' '*2 + str(int(fp/valorN)) +  ' '*7 + str(int(tn/valorN))  + '\n'
  s += '\n'

  # adiciono os valores de cada métrica a minha string de retorno com MulticlassClassificationEvaluator
  evaluator = MulticlassClassificationEvaluator()

  s += f'Acurácia: {evaluator.evaluate(df_transform_modelo, {evaluator.metricName: "accuracy"})}\n'
  s += f'Precisão: {evaluator.evaluate(df_transform_modelo, {evaluator.metricName: "precisionByLabel", evaluator.metricLabel: 1})}\n'
  s += f'Recall: {evaluator.evaluate(df_transform_modelo, {evaluator.metricName: "recallByLabel", evaluator.metricLabel: 1})}\n'
  s += f'F1: {evaluator.evaluate(df_transform_modelo, {evaluator.metricName: "fMeasureByLabel", evaluator.metricLabel: 1})}\n'

  return s

In [76]:
calcula_mostra_metricas_evaluate(previsoes_dtc_teste, normalize=False)

'                    Previsto\n               Churn     Não-Churn\n    Churn      1181       382\nReal\n    Não-Churn  336       1243\n\nAcurácia: 0.7714831317632082\nPrecisão: 0.7785102175346078\nRecall: 0.7555982085732565\nF1: 0.7668831168831168\n'

# Random Forest Classifier

##Ajuste e previsão

In [77]:
from pyspark.ml.classification import RandomForestClassifier

In [78]:
rfc = RandomForestClassifier(seed=SEED)

In [80]:
modelo_rfc = rfc.fit(treino)

In [81]:
previsoes_rfc_treino = modelo_rfc.transform(treino)

In [82]:
previsoes_rfc_teste = modelo_rfc.transform(teste)

In [83]:
previsoes_rfc_teste.show()

+--------------------+-----+--------------------+--------------------+----------+
|            features|label|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|(24,[0,1,2,3,4,5,...|    0|[16.7433871675615...|[0.83716935837807...|       0.0|
|(24,[0,1,2,3,4,5,...|    0|[7.27313214599648...|[0.36365660729982...|       1.0|
|(24,[0,1,2,3,4,5,...|    1|[7.46885072161585...|[0.37344253608079...|       1.0|
|(24,[0,1,2,3,4,5,...|    1|[9.33276328267787...|[0.46663816413389...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[7.79829004739264...|[0.38991450236963...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[7.13263407834549...|[0.35663170391727...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[4.45872635511159...|[0.22293631775557...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[7.84691519125130...|[0.39234575956256...|       1.0|
|(24,[0,1,2,3,4,5,...|    0|[9.94796150783366...|[0.49739807539168...|       1.0|
|(24,[0,1,2,3,4,

##Métricas

Criando uma função

In [90]:
def metricas_calc(df_transform_modelo):
  print("Acurácia: %f" % evaluator.evaluate(df_transform_modelo, {evaluator.metricName: 'accuracy'}))
  print("Precisão: %f" % evaluator.evaluate(df_transform_modelo, {evaluator.metricName: 'precisionByLabel'}))
  print("Recall: %f" % evaluator.evaluate(df_transform_modelo, {evaluator.metricName: 'recallByLabel'}))
  print("F1: %f" % evaluator.evaluate(df_transform_modelo, {evaluator.metricName: 'fMeasureByLabel'}))

In [100]:
def compara_metricas(modelo_x, modelo_y):
  if evaluator.evaluate(modelo_x , {evaluator.metricName: 'accuracy'})> evaluator.evaluate(modelo_y, {evaluator.metricName: 'accuracy'}):
    print(f'O primeiro modelo teve melhor resultado')
  else:
    print(f'O segundo modelo teve melhor resultado')

Treino

In [91]:
metricas_calc(previsoes_rfc_treino)

Acurácia: 0.785595
Precisão: 0.803974
Recall: 0.754103
F1: 0.778240


Teste

In [92]:
metricas_calc(previsoes_rfc_teste)

Acurácia: 0.770210
Precisão: 0.791695
Recall: 0.736542
F1: 0.763123


# Cross Validation (CV)

##Árvore de decisão com CV

In [102]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [103]:
dtc = DecisionTreeClassifier(seed=SEED)

ParamGridBuilder(): Inicia o processo de construção de uma grade de parâmetros. Esta grade define as combinações de valores de hiperparâmetros que serão testadas para otimizar o modelo.


.addGrid(dtc.maxDepth, [2, 5, 10]): Adiciona o hiperparâmetro maxDepth (profundidade máxima da árvore) à grade. O modelo será treinado com árvores de profundidade 2, 5 e 10.



.addGrid(dtc.maxBins, [10, 32, 45]): Adiciona o hiperparâmetro maxBins (número máximo de "bins" para discretização de recursos contínuos) à grade. O modelo será treinado com 10, 32 e 45 bins.



.build(): Finaliza a construção da grade, criando todas as combinações possíveis desses hiperparâmetros. Por exemplo, uma das combinações será maxDepth=2 e maxBins=10, outra será maxDepth=5 e maxBins=32, e assim por diante.



Esta grade será usada posteriormente com um CrossValidator para encontrar a melhor combinação de hiperparâmetros para o modelo de Árvore de Decisão, com base em alguma métrica de avaliação.

In [104]:
grid = ParamGridBuilder()\
.addGrid(dtc.maxDepth, [2, 5, 10])\
.addGrid(dtc.maxBins, [10, 32, 45])\
.build()

In [105]:
evaluator = MulticlassClassificationEvaluator()

Este código está configurando um objeto CrossValidator (Validação Cruzada) para otimizar seu modelo de Árvore de Decisão. Vamos entender cada parte:

estimator=dtc: Define o estimador (o modelo) que será avaliado e otimizado. Neste caso, é a DecisionTreeClassifier (dtc) que você já instanciou.


estimatorParamMaps=grid: Fornece a grade de hiperparâmetros (grid) que você criou anteriormente usando o ParamGridBuilder. O CrossValidator irá testar todas as combinações de maxDepth e maxBins definidas nesta grade.


evaluator=evaluator: Especifica a métrica de avaliação que será usada para determinar a performance de cada combinação de hiperparâmetros. Você definiu um MulticlassClassificationEvaluator para isso.


numFolds=3: Indica o número de 'folds' (dobras) para a validação cruzada. Isso significa que o conjunto de dados de treino será dividido em 3 partes, e o modelo será treinado 3 vezes, usando 2 partes para treino e 1 para validação em cada iteração.


seed=SEED: Define uma semente para a geração de números aleatórios, garantindo que os resultados da divisão dos dados e do treinamento sejam reproduzív

In [106]:
dtc_cv = CrossValidator(
    estimator=dtc,
    estimatorParamMaps=grid,
    evaluator=evaluator,
    numFolds=3,
    seed=SEED
)


In [107]:
modelo_dtc_cv = dtc_cv.fit(treino)

In [109]:
previsoes_dtc_cv_teste = modelo_dtc_cv.transform(teste)

In [110]:
metricas_calc(previsoes_dtc_cv_teste)

Acurácia: 0.790261
Precisão: 0.830460
Recall: 0.732109
F1: 0.778189


#Random Forest com CV

In [113]:
rfc = RandomForestClassifier(seed = SEED)

In [114]:
grid = ParamGridBuilder()\
.addGrid(rfc.maxDepth, [2, 5, 10])\
.addGrid(rfc.maxBins, [10,32, 45])\
.addGrid(rfc.numTrees, [10, 20, 50])\
.build()

In [116]:
evaluator = MulticlassClassificationEvaluator()

In [117]:
rfc_cv = CrossValidator(
    estimator=rfc,
    estimatorParamMaps=grid,
    evaluator=evaluator,
    numFolds = 3,
    seed = SEED)

In [118]:
modelo_rfc_cv = rfc_cv.fit(treino)

In [119]:
previsoes_rfc_cv_teste = modelo_rfc_cv.transform(teste)

In [120]:
metricas_calc(previsoes_rfc_cv_teste)

Acurácia: 0.812858
Precisão: 0.836388
Recall: 0.780241
F1: 0.807339


#Modelo Final

In [122]:
melhor_modelo_rfc_cv = modelo_rfc_cv.bestModel

vendo hiperparametros

In [124]:
print(melhor_modelo_rfc_cv.getMaxDepth())
print(melhor_modelo_rfc_cv.getMaxBins())
print(melhor_modelo_rfc_cv.getNumTrees)

10
45
20


In [127]:
rfc_tunning= RandomForestClassifier(maxDepth=10, maxBins=45, numTrees=20, seed=SEED)

pegando os dados preparados

In [128]:
modelo_rfc_tunning = rfc_tunning.fit(dataset_prep)

In [130]:
spark.stop()